In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_british_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [5]:
british_dataset = read_file("../Dataset/british_english_sms_corpus.txt")
# for i in british_dataset:
#     print(i)
# british_dataset.index('SMS Ham\n')
# british_dataset.index('SMS Spam\n')
british_ham = [i[:-1] for i in british_dataset[17:480] if i!='\n']
british_spam = [i[:-1] for i in british_dataset[482:] if i!='\n']

british_dataset = {'ham': british_ham, 'spam': british_spam}

print(len(british_dataset['ham']))
print(len(british_dataset['spam']))

450
425


In [6]:
british_dataset = transform_british_dict(british_dataset)
british_dataset.head()

,message,label
0,Oi when you gonna ring,ham
1,"Oh yes, why is it like torture watching england?",ham
2,Yes but can we meet in town cos will go to gep...,ham
3,what I meant to say is cant wait to see u agai...,ham
4,V nice! Off 2 sheffield tom 2 air my opinions ...,ham


In [7]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'british Dataset_'+'.csv')['URL'].to_list())

In [8]:
british_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'british Dataset_'+'.csv')['URL']
british_dataset['Message Len'] = [len(i) for i in british_dataset['message']]
british_dataset.head()

,message,label,Extracted URL,Message Len
0,Oi when you gonna ring,ham,NaN,22
1,"Oh yes, why is it like torture watching england?",ham,NaN,48
2,Yes but can we meet in town cos will go to gep...,ham,NaN,141
3,what I meant to say is cant wait to see u agai...,ham,NaN,87
4,V nice! Off 2 sheffield tom 2 air my opinions ...,ham,NaN,127


In [9]:
british_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'british Websites Analysis'+'.csv')
british_website_analysis_data = british_website_analysis_data.drop(columns=['ham', 'spam'])
british_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,www.B4Utele.com,www.B4Utele.com,0,0,-1,0
1,www.Ldew.com.subs16+1win150ppmx3,www.Ldew.com.subs16+1win150ppmx3.,0,0,-1,0
2,www.SMS.ac/u/goldviking,www.SMS.ac,1034,0,200,1
3,http://www.vouch4me.com/etlp/dining.asp,www.vouch4me.com,0,0,-1,0
4,www.4-tc.biz,www.4-tc.biz,0,0,-1,0


In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
british_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24388\3068849319.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  british_website_analysis_data.iloc[0][0]


'www.B4Utele.com'

In [12]:
# for row in enron_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = british_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(british_website_analysis_data['FQDN'])}
website_data = british_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
british_dataset['FQDN'] = fqdn
british_dataset['Website Size in KB'] = website_size
british_dataset['Website Textual Content Length'] = text_content_len
british_dataset['Status Code'] = status_code
british_dataset['Parked'] = parked

In [16]:
british_dataset = british_dataset.replace('', np.nan)
british_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24388\3205473444.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  british_dataset = british_dataset.replace('', np.nan)


,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,Oi when you gonna ring,ham,NaN,22,NaN,NaN,NaN,NaN,NaN
1,"Oh yes, why is it like torture watching england?",ham,NaN,48,NaN,NaN,NaN,NaN,NaN
2,Yes but can we meet in town cos will go to gep...,ham,NaN,141,NaN,NaN,NaN,NaN,NaN
3,what I meant to say is cant wait to see u agai...,ham,NaN,87,NaN,NaN,NaN,NaN,NaN
4,V nice! Off 2 sheffield tom 2 air my opinions ...,ham,NaN,127,NaN,NaN,NaN,NaN,NaN


In [17]:
# Counter(british_dataset['FQDN'].to_list())
british_dataset[(british_dataset['Extracted URL'].notna()) & (british_dataset['FQDN'].isna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(british_dataset['FQDN'].to_list())
british_dataset[(british_dataset['Extracted URL'].notna()) & (british_dataset['FQDN'].notna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
452,goldviking (29/M) is inviting you to be his fr...,spam,www.SMS.ac/u/goldviking,138,www.SMS.ac,1034.0,0.0,200.0,1.0
454,Santa calling! Would your little ones like a c...,spam,www.santacalling.com,158,www.santacalling.com,0.0,0.0,-1.0,0.0
459,"Text82228>> Get more ringtones, logos and game...",spam,www.txt82228.com,101,www.txt82228.com,0.0,0.0,-1.0,0.0
464,"SMSSERVICES. for yourinclusive text credits, p...",spam,www.comuk.net,156,www.comuk.net,0.0,0.0,-1.0,0.0
480,For ur chance to win a £250 wkly shopping spre...,spam,www.txt-2-shop.com,126,www.txt-2-shop.com,0.0,0.0,-1.0,0.0
509,FREE entry into our £250 weekly comp just send...,spam,www.textcomp.com,122,www.textcomp.com,0.0,0.0,-1.0,0.0
514,"SMS SERVICES. for your inclusive text credits,...",spam,www.comuk.net,158,www.comuk.net,0.0,0.0,-1.0,0.0
522,-PLS STOP bootydelious (32/F) is inviting you ...,spam,www.SMS.ac/u/bootydelious,152,www.SMS.ac,1034.0,0.0,200.0,1.0
539,For taking part in our mobile survey yesterday...,spam,www.txt43.com,156,www.txt43.com,114.0,0.0,200.0,1.0
542,"Dear Voucher Holder, To claim this weeks offer...",spam,http://www.e-tlp.co.uk/expressoffer,152,www.e-tlp.co.uk,0.0,0.0,-1.0,0.0


In [19]:
print(len(british_dataset))

875


In [26]:
#messages with URL
print(len(british_dataset[(british_dataset['Extracted URL'].notna())]), len(british_dataset[(british_dataset['Extracted URL'].notna())])/len(british_dataset))

55 0.06285714285714286


In [27]:
#spam messages with URL
print(len(british_dataset[(british_dataset['Extracted URL'].notna()) & (british_dataset['label']=='spam')]), len(british_dataset[(british_dataset['Extracted URL'].notna()) & (british_dataset['label']=='spam')])/len(british_dataset[british_dataset['label']=='spam']))

55 0.12941176470588237


In [22]:
#unique FQDN
len(set(british_dataset[(british_dataset['FQDN'].notna())]['FQDN']))

41

In [23]:
only_unique_live_websites_data = british_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='ham')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='spam')]))

7
0
7


In [24]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='ham')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='spam')]))

4
0
4


In [25]:
british_dataset.to_csv('../Dataset/Refined_English_SMS_Dataset.csv', index=None)